In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import cv2
import glob 

from data_utils import SegmentationData, label_img_to_rgb, get_train_transforms, compute_class_weights
from file_loader import get_all_files
from network.UNet import UNet
#from network.MultiResUNet import MultiResUNet
from create_borders import create_border_mask
from seg_utils import _find_contours, combining_predicted_segmented_and_border,select_cell_contour

#set up default cuda device
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# for auto-reloading external modules
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [ ]:
data = 'labelled_test_data/images'
files = get_all_files(data, 'png')
len(files)

In [ ]:
!python test.py

### To get mean and std of test images from CDB and EH5 RBC analysis samples

In [ ]:
test_data = TestData(image_paths_file='labelled_test_data/test.txt')
test_loader = torch.utils.data.DataLoader(test_data,
                                          batch_size=5,
                                          shuffle=False,
                                          num_workers=1)
nimages = 0
mean = 0.0
var = 0.0
for inputs in test_loader:#inputs is tensor with values between (0,1) - to calculate mean and std
    batch = inputs
    # Rearrange batch to be the shape of [B, C, W * H]
    batch = batch.view(batch.size(0), batch.size(1), -1)
    nimages += batch.size(0)
    # Compute mean and std here
    mean += batch.mean(2).sum(0) 
    var += batch.var(2).sum(0)
mean /= nimages
var /= nimages
std = torch.sqrt(var)
print(mean)
print(std)

In [ ]:
import torch.utils.data as data
from PIL import Image
from torchvision import transforms
import _pickle as pickle


SEG_LABELS_LIST = [
   
    {"id": 0,  "name": "Background",   "rgb_values": [  0,   0,    0]},
    {"id": 1,  "name": "Whitening_R",  "rgb_values": [ 95,  95,   95]},
    {"id": 1,  "name": "Whitening",    "rgb_values": [255, 255,  255]},
    {"id": 1,  "name": "Whitening2",   "rgb_values": [128, 128,  128]},
    {"id": 1,  "name": "Whitening3",   "rgb_values": [225, 225,  225]},
    {"id": 1,  "name": "Hemoglobin",   "rgb_values": [220,  20,   60]},
    {"id": 1,  "name": "Hemoglobin2",  "rgb_values": [165,  42,   42]},
    
    {"id": 1,  "name": "Platelet",     "rgb_values": [240, 170,  215]},
    {"id": 1,  "name": "Nucleus",      "rgb_values": [ 70, 130,  180]},
    {"id": 1,  "name": "Rna",          "rgb_values": [ 10,   0,  180]},
    {"id": 1,  "name": "Reticulocyte", "rgb_values": [100, 238,  238]},
    {"id": 1,  "name": "WBC",          "rgb_values": [150,   0,  170]},
    {"id": 1,  "name": "Hemoglobin3",  "rgb_values": [238, 105,   43]}]

def label_img_to_rgb(label_img):
    label_img = np.squeeze(label_img)
    labels = np.unique(label_img)
    label_infos = [l for l in SEG_LABELS_LIST if l['id'] in labels]

    label_img_rgb = np.array([label_img,
                              label_img,
                              label_img]).transpose(1,2,0)  #(H,W,C)
    for l in label_infos:
        mask = label_img == l['id']
        label_img_rgb[mask] = l['rgb_values']

    return label_img_rgb.astype(np.uint8)



class TestData(data.Dataset):

    def __init__(self, image_paths_file):
        with open(image_paths_file) as f:
            self.image_names = f.read().splitlines()
            
            
    def __getitem__(self, key):
        if isinstance(key, slice):
            # get the start, stop, and step from the slice
            return [self[ii] for ii in range(*key.indices(len(self)))]
        elif isinstance(key, int):
            # handle negative indices
            if key < 0:
                key += len(self)
            if key < 0 or key >= len(self):
                raise IndexError("The index (%d) is out of range." % key)
            # get the data from direct index
            return self.get_item_from_index(key)
        else:
            raise TypeError("Invalid argument type.")

    def __len__(self):
        return len(self.image_names)
    
    
    def get_item_from_index(self, index):
        
        to_tensor = transforms.Compose( [transforms.ToTensor(),
                                         transforms.Normalize((0.6843, 0.6203, 0.6554), (0.0837, 0.1290, 0.0753))])

        image_path = self.image_names[index]
        
        img = cv2.imread(image_path)   
        img =cv2.cvtColor(img, cv2.COLOR_BGR2RGB) ## opencv reads the color channels in reverse order :(
        
        img = to_tensor(img)          
            
        return img


### Semantic Mask Predictions

In [ ]:
model = torch.load("models/unet_50_rs_2_c_2_no_test_final.model")
test_data = TestData(image_paths_file='labelled_test_data/test.txt')
test_loader = torch.utils.data.DataLoader(test_data,
                                          batch_size=1,
                                          shuffle=False,
                                          num_workers=1)

predictions = []
model.eval()
for inputs in test_loader:
    inputs = inputs.to(device)
    
    outputs = model.forward(inputs)
    _, preds = torch.max(outputs, 1)
    
    #saving the predictions to numpy array
    pred = preds[0].data.cpu()
    pred = pred.numpy()
    predictions.append(pred)
    
model.train()
predictions = np.array(predictions)

#save the predicted segmented images
test_files = np.loadtxt('labelled_test_data/test.txt', dtype = str)
    
assert len(predictions) == len(test_files)
    
for i, file in enumerate(test_files,0):
        file = file.replace('images','test_predictions/segmented_images_latest')
        out_dir, img = os.path.split(file)
        
        
        if not os.path.isdir(out_dir):
            os.makedirs(out_dir)
        
        prediction = label_img_to_rgb(predictions[i])   #this is RGB image
        prediction = cv2.cvtColor(prediction, cv2.COLOR_RGB2BGR) # conversion to BGR
        cv2.imwrite(file,prediction);

### Border Mask Predictions

In [ ]:
model2 = torch.load("models/unet_border_50_rs_2_platelet_no_test_final.model")

border_predictions = []
model2.eval()
for inputs in test_loader:
    inputs = inputs.to(device)
    
    outputs = model2.forward(inputs)
    _, preds = torch.max(outputs, 1)


    #saving the border predictions to numpy array
    pred = preds[0].data.cpu()
    pred = pred.numpy()
    border_predictions.append(pred)
    
model2.train()
border_predictions = np.array(border_predictions)
#save the predicted border images
test_files = np.loadtxt('labelled_test_data/test.txt', dtype = str)
for i, file in enumerate(test_files,0):
        file = file.replace('images','test_predictions/border_images_latest')
        out_dir, img = os.path.split(file)
        
        
        if not os.path.isdir(out_dir):
            os.makedirs(out_dir)
        
        prediction = label_img_to_rgb(border_predictions[i])   
        prediction = cv2.cvtColor(prediction, cv2.COLOR_RGB2BGR)
        cv2.imwrite(file,prediction);


### Get only center cell mask (Post Processing step)

In [ ]:
data = 'labelled_test_data/test_predictions/border_images_latest/'
border_images = get_all_files(data, 'png')

data = 'labelled_test_data/test_predictions/segmented_images_latest/'
segmented_images = get_all_files(data, 'png')

center_cell_images = combining_predicted_segmented_and_border(border_images, segmented_images)

#save_test_predictions_combined(center_cell_images)
data = 'labelled_test_data/test_predictions/border_images_latest/'
border_images = get_all_files(data, 'png')
    
    
for i, file in enumerate(border_images,0):
        file = file.replace('border_images_latest','combined_latest')
        out_dir, img = os.path.split(file)
        
        if not os.path.isdir(out_dir):
            os.makedirs(out_dir)
            
        cv2.imwrite(file,center_cell_images[i]); #predictions are in BGR format

In [ ]:
center_mask_folder = 'labelled_test_data/test_predictions/combined_latest/'
center_mask_paths = get_all_files(center_mask_folder, 'png')
len(center_mask_paths)

### Get the center cell from input image using center cell mask

In [ ]:

for mask_path in center_mask_paths:
    color_masked_img = cv2.imread(mask_path)
    masked_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    contours_cell = _find_contours(masked_img)   #covers the whole cell(fills gap if any), contour RETR_EXTERNAL method
    if (len(contours_cell) != 1 ):
        print(mask_path)
    assert(len(contours_cell) == 1 )
    cell_of_interest_contour = np.zeros(masked_img.shape, masked_img.dtype)
    cv2.drawContours(cell_of_interest_contour, contours_cell, -1, 255, -1)
    

        
    image_path = mask_path.replace('test_predictions/combined_latest','images')
    
    #Read the corresponding test input image
    input_img = cv2.imread(image_path)
    
    
    #Pick only cell of interest from input image
    input_img[cell_of_interest_contour<255] = 0
    
    
    out_path = image_path.replace('images','test_center_cell_images_latest')
    
    
    out_dir, img = os.path.split(out_path)
    if not os.path.isdir(out_dir):
        os.makedirs(out_dir)
       
    #Write the center cell of test image
   
    cv2.imwrite(out_path, input_img)
    


In [ ]:
center_cell_folder = 'labelled_test_data/test_center_cell_images_latest'
center_cell_paths = get_all_files(center_cell_folder, 'png')
len(center_cell_paths)